# Module 3 : Aspects et Sentiment par Aspect

Ce notebook implémente le **Module 3** de ReviewScope :
1. Segmentation des avis en phrases (`split_sentences`).
2. Détection d'aspects critiques (prix, emballage, goût, livraison, fraîcheur, santé, qualité, quantité) :
   - Approche 1 : Dictionnaire de mots-clés
   - Approche 2 : Expansion sémantique par similarité d'embeddings (`all-MiniLM-L6-v2`)
3. Attribution d'une polarité de sentiment par aspect via le classifieur entraîné au Module 1.
4. Construction de la matrice : **Avis x Aspect x Polarité**.
5. Comparaison méthodologique et validation sur échantillon annoté.


In [9]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data_loader import stratified_sample
from src.aspects import (
    split_sentences,
    detect_aspects_keywords,
    detect_aspects_embeddings,
    build_aspect_matrix,
    summarize_aspects
)
from src.utils.visualization import plot_aspect_summary


## 1. Chargement d'un sous-ensemble représentatif


In [3]:
clean_path = config.DATA_PROCESSED / "reviews_clean.csv"
df = pd.read_csv(clean_path)
df_sample = stratified_sample(df, n=5000, stratify_col="Score", random_state=config.RANDOM_STATE)
print(f"Échantillon chargé : {len(df_sample)} avis")


Échantillon chargé : 5000 avis


## 2. Découpage en phrases et Dictionnaire de mots-clés


In [4]:
sample_text = df_sample.iloc[0]["Text"]
print("Texte exemple :", sample_text)
sentences = split_sentences(sample_text)
print(f"\nPhrases découpées ({len(sentences)}) :")
for s in sentences:
    aspects = detect_aspects_keywords(s)
    print(f"  - [{', '.join(aspects) if aspects else 'aucun'}] {s}")


Texte exemple : OK, so one bag isn't ever going to be enough. These are really, really good. I've shared them with friends and everyone that tries them loves them. Great stuff!

Phrases découpées (4) :
  - [quantite] OK, so one bag isn't ever going to be enough.
  - [qualite] These are really, really good.
  - [aucun] I've shared them with friends and everyone that tries them loves them.
  - [qualite] Great stuff!


## 3. Construction de la Matrice Avis x Aspect x Sentiment (Dictionnaire)


In [5]:
print("Construction de la matrice d'aspects...")
aspect_df = build_aspect_matrix(df_sample, method="keywords")
print(f"Total phrases avec aspect identifié : {len(aspect_df):,}")
print(aspect_df.head(10))


Construction de la matrice d'aspects...
Total phrases avec aspect identifié : 18,910
   review_idx                                           sentence     aspect  \
0           0      OK, so one bag isn't ever going to be enough.   quantite   
1           0                     These are really, really good.    qualite   
2           0                                       Great stuff!    qualite   
3           1  My dogs can't get enough of these healthy snacks.      sante   
4           1  My dogs can't get enough of these healthy snacks.   quantite   
5           2  Light, clean flavor of olives with a peppery/f...       gout   
6           3  me: Normally drink Rainforest Espresso Tried t...       prix   
7           3  me: Normally drink Rainforest Espresso Tried t...   quantite   
8           3  Very good and love the concept but wish they s...  emballage   
9           3  Very good and love the concept but wish they s...  livraison   

  review_sentiment  
0          positif  
1  

## 4. Résumé et Visualisation des Aspects et Sentiments


In [6]:
summary = summarize_aspects(aspect_df)
print("Résumé des aspects :\n", summary)
plot_aspect_summary(summary)


Résumé des aspects :
            count  pct_negative  pct_positive
aspect                                      
gout        5138      0.141300      0.766251
qualite     3820      0.116492      0.803665
quantite    2838      0.143058      0.760042
sante       2027      0.120868      0.794277
prix        1767      0.140917      0.772496
livraison   1554      0.161519      0.753539
emballage   1272      0.210692      0.695755
fraicheur    494      0.101215      0.809717
Figure sauvegardée : C:\Users\loanr\Documents\UQAC\cours\aa\projet_final\projet_aa\reports\figures\aspect_summary.png


<Figure size 1400x600 with 2 Axes>

## 5. Détection par Embeddings (Similarité Sémantique)


In [7]:
test_sentence = "The box arrived crushed and the bag inside was torn open."
emb_aspects = detect_aspects_embeddings(test_sentence)
print(f"Phrase : '{test_sentence}'")
print("Aspects détectés par embeddings (similarité cosinus) :")
for asp, sim in emb_aspects:
    print(f"  - {asp}: {sim:.4f}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Phrase : 'The box arrived crushed and the bag inside was torn open.'
Aspects détectés par embeddings (similarité cosinus) :
  - emballage: 0.4797


## 6. Sauvegarde de la matrice d'aspects pour le Module 4 et 6


In [8]:
output_path = config.DATA_PROCESSED / "aspects_sample.csv"
aspect_df.to_csv(output_path, index=False)
print(f"Matrice d'aspects sauvegardée dans {output_path}")


Matrice d'aspects sauvegardée dans C:\Users\loanr\Documents\UQAC\cours\aa\projet_final\projet_aa\data\processed\aspects_sample.csv
